<a href="https://colab.research.google.com/github/data4class/Teaching/blob/main/Colluding_set_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.cluster import SpectralClustering
import matplotlib.pyplot as plt

# Step 1: Load and preprocess the data
file_id = '1hctEm-25IIt9CPUduQExa1COSyS9o7_G'
url = f'https://drive.google.com/uc?id={file_id}&export=download'

# Load with pipe separator and no header
df = pd.read_csv(url, sep='|', header=None)

# Assign proper column names (0-indexed)
# Column 4 (index 3): price, Column 5 (index 4): volume
# Column 7 (index 6): buy_id, Column 8 (index 7): sell_id
df.columns = ['col0', 'col1', 'col2', 'price', 'volume', 'col5', 'buy_id', 'sell_id']

print("Data loaded successfully!")
print("Column names:", df.columns.tolist())
print("First few rows:")
print(df.head())
print(f"Dataset shape: {df.shape}")

# Step 2: Extract first 7 characters of trader IDs
df['buy_trader'] = df['buy_id'].astype(str).str[:7]
df['sell_trader'] = df['sell_id'].astype(str).str[:7]

print(f"\nSample data:")
print(f"Price (column 4): {df['price'].head().tolist()}")
print(f"Volume (column 5): {df['volume'].head().tolist()}")
print(f"Buy ID (column 7): {df['buy_id'].head().tolist()}")
print(f"Sell ID (column 8): {df['sell_id'].head().tolist()}")
print(f"Buy traders (first 7 chars): {df['buy_trader'].head().tolist()}")
print(f"Sell traders (first 7 chars): {df['sell_trader'].head().tolist()}")

# Check for unique traders
print(f"\nTrader Statistics:")
print(f"Unique buy traders: {df['buy_trader'].nunique()}")
print(f"Unique sell traders: {df['sell_trader'].nunique()}")
all_traders = set(df['buy_trader']) | set(df['sell_trader'])
print(f"Total unique traders: {len(all_traders)}")

# Step 3: Create a weighted graph
G = nx.Graph()
transaction_count = 0
skipped_same_trader = 0

for _, row in df.iterrows():
    price = row['price']
    volume = row['volume']
    buy_trader = row['buy_trader']
    sell_trader = row['sell_trader']

    # Skip if same trader
    if buy_trader == sell_trader:
        skipped_same_trader += 1
        continue

    # Skip if any values are NaN
    if pd.isna(price) or pd.isna(volume):
        continue
    # Ex: Include other parameters such as number transactions, price movement etc.
    weight = price * volume
    transaction_count += 1

    if G.has_edge(buy_trader, sell_trader):
        G[buy_trader][sell_trader]['weight'] += weight
    else:
        G.add_edge(buy_trader, sell_trader, weight=weight)

print(f"\nGraph Creation:")
print(f"Total transactions processed: {transaction_count}")
print(f"Transactions skipped (same trader): {skipped_same_trader}")
print(f"Graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

# Show some sample edges with weights
sample_edges = list(G.edges(data=True))[:5]
print(f"\nSample edges with weights:")
for u, v, d in sample_edges:
    print(f"  {u} <-> {v}: {d['weight']:,.0f}")

# Check if graph has enough nodes for clustering
if G.number_of_nodes() < 2:
    print("Error: Graph has fewer than 2 nodes. Cannot perform clustering.")
    exit()

# Normalize weights to be between 0 and 1
max_weight = max([d['weight'] for u, v, d in G.edges(data=True)])
for u, v, d in G.edges(data=True):
    d['weight'] = d['weight'] / max_weight

print(f"Weights normalized. Max weight was: {max_weight:,.0f}")


In [ ]:

# Step 4: Apply spectral clustering
# Dynamic cluster selection based on graph size
n_nodes = G.number_of_nodes()
n_clusters = min(10, max(2, n_nodes // 50))  # Reasonable number of clusters

print(f"\nClustering with {n_clusters} clusters for {n_nodes} nodes")

adj_matrix = nx.to_numpy_array(G)
sc = SpectralClustering(n_clusters=n_clusters, affinity='precomputed', n_init=100, random_state=42)
labels = sc.fit_predict(adj_matrix)

# Step 5: Analyze the results
nodes = list(G.nodes())
clusters = {}
for i in range(n_clusters):
    clusters[i] = [nodes[j] for j in range(len(nodes)) if labels[j] == i]

print(f"\nClustering Results:")
for i, cluster in clusters.items():
    print(f"Cluster {i}: {len(cluster)} nodes")



In [ ]:
# Select Cluster 3 for analysis (change it for others)
target_cluster_id = 3
target_cluster = clusters[target_cluster_id]

print(f"\nAnalyzing Cluster {target_cluster_id}:")
print(f"Cluster {target_cluster_id} has {len(target_cluster)} nodes")
print(f"Traders in Cluster {target_cluster_id}: {target_cluster}")

# Create a subgraph of Cluster 3
subgraph = G.subgraph(target_cluster)

# Get the largest connected component from Cluster 3
final_subgraph = subgraph
if subgraph.number_of_nodes() > 0:
    connected_components = list(nx.connected_components(subgraph))
    if connected_components:
        largest_cc = max(connected_components, key=len)
        final_subgraph = subgraph.subgraph(largest_cc)
        print(f"Largest connected component in Cluster {target_cluster_id}: {len(largest_cc)} nodes")
        if len(connected_components) > 1:
            print(f"Note: Cluster has {len(connected_components)} disconnected components")
            for i, comp in enumerate(sorted(connected_components, key=len, reverse=True)):
                print(f"  Component {i+1}: {len(comp)} nodes")
    else:
        print("No connected components found in Cluster 3")
else:
    print("Cluster 3 is empty")

# Print the trader IDs of Cluster 3
print(f"\nCluster {target_cluster_id} Analysis (Largest Connected Component):")
print(f"Number of traders: {final_subgraph.number_of_nodes()}")
print(f"Number of trading relationships: {final_subgraph.number_of_edges()}")
print("All Trader IDs in this cluster:")
print(list(final_subgraph.nodes()))

# Print detailed statistics for Cluster 3
print(f"\nDetailed Statistics for Cluster {target_cluster_id}:")
print(f"Original graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Cluster {target_cluster_id}: {final_subgraph.number_of_nodes()} nodes, {final_subgraph.number_of_edges()} edges")

# Calculate clustering coefficient and other network metrics
if final_subgraph.number_of_nodes() > 1:
    density = nx.density(final_subgraph)
    avg_clustering = nx.average_clustering(final_subgraph)
    print(f"Network density: {density:.4f}")
    print(f"Average clustering coefficient: {avg_clustering:.4f}")

    # Degree centrality
    degree_centrality = nx.degree_centrality(final_subgraph)
    print(f"Most central traders (by degree):")
    sorted_centrality = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)
    for trader, centrality in sorted_centrality:
        degree = final_subgraph.degree[trader]
        print(f"  {trader}: degree={degree}, centrality={centrality:.3f}")

# Print weight statistics for Cluster 3
weights = [d['weight'] for u, v, d in final_subgraph.edges(data=True)]
if weights:
    print(f"\nNormalized weights in Cluster {target_cluster_id}:")
    print(f"  Range: [{min(weights):.6f}, {max(weights):.6f}]")
    print(f"  Average: {np.mean(weights):.6f}")
    print(f"  Standard deviation: {np.std(weights):.6f}")
else:
    print(f"No edges in Cluster {target_cluster_id}.")

# Step 6: Visualize Cluster 3
if final_subgraph.number_of_nodes() > 0:
    plt.figure(figsize=(16, 12))

    # Use different layout algorithms for better visualization
    if final_subgraph.number_of_nodes() <= 15:
        pos = nx.spring_layout(final_subgraph, seed=42, k=3, iterations=100)
    else:
        pos = nx.kamada_kawai_layout(final_subgraph)

    # Calculate node sizes based on degree centrality
    degrees = dict(final_subgraph.degree())
    max_degree = max(degrees.values()) if degrees else 1
    node_sizes = [800 + (degrees[node] / max_degree) * 1200 for node in final_subgraph.nodes()]

    # Color nodes based on their degree (more connected = darker)
    node_colors = [degrees[node] for node in final_subgraph.nodes()]

    # Draw nodes
    nodes = nx.draw_networkx_nodes(final_subgraph, pos,
                                  node_color=node_colors,
                                  node_size=node_sizes,
                                  cmap=plt.cm.Blues_r,
                                  alpha=0.8)

    # Draw labels
    nx.draw_networkx_labels(final_subgraph, pos, font_size=10, font_weight='bold')

    # Draw edges with width proportional to weight
    if final_subgraph.number_of_edges() > 0:
        edge_weights = [d['weight'] for (u, v, d) in final_subgraph.edges(data=True)]
        max_edge_weight = max(edge_weights)
        min_edge_weight = min(edge_weights)

        # Normalize edge widths between 1 and 8
        if max_edge_weight > min_edge_weight:
            edge_widths = [1 + 7 * (w - min_edge_weight) / (max_edge_weight - min_edge_weight)
                          for w in edge_weights]
        else:
            edge_widths = [4] * len(edge_weights)

        # Color edges by weight
        edges = nx.draw_networkx_edges(final_subgraph, pos,
                                      width=edge_widths,
                                      edge_color=edge_weights,
                                      edge_cmap=plt.cm.Reds,
                                      alpha=0.7)

        # Add colorbar for edges
        if edges is not None:
            plt.colorbar(edges, label='Normalized Trading Volume', shrink=0.8)

        # Add edge labels for small graphs
        if final_subgraph.number_of_edges() <= 15:
            edge_labels = nx.get_edge_attributes(final_subgraph, 'weight')
            edge_labels = {k: f'{v:.3f}' for k, v in edge_labels.items()}
            nx.draw_networkx_edge_labels(final_subgraph, pos, edge_labels=edge_labels,
                                       font_size=8, bbox=dict(boxstyle='round,pad=0.2',
                                                             facecolor='white', alpha=0.7))

    # Add colorbar for nodes
    if nodes is not None:
        plt.colorbar(nodes, label='Node Degree', shrink=0.8, pad=0.1)

    plt.title(f"Trading Network - Cluster {target_cluster_id}\n"
              f"({final_subgraph.number_of_nodes()} traders, "
              f"{final_subgraph.number_of_edges()} relationships)",
              fontsize=16, fontweight='bold', pad=20)

    plt.axis('off')
    plt.tight_layout()
    plt.show()

    # Create a second plot showing the full cluster (all components) if there are multiple
    if subgraph.number_of_nodes() != final_subgraph.number_of_nodes():
        plt.figure(figsize=(14, 10))
        pos_full = nx.spring_layout(subgraph, seed=42, k=2, iterations=50)

        # Identify different components with different colors
        components = list(nx.connected_components(subgraph))
        colors = plt.cm.Set3(np.linspace(0, 1, len(components)))

        for i, component in enumerate(components):
            component_nodes = list(component)
            nx.draw_networkx_nodes(subgraph, pos_full,
                                  nodelist=component_nodes,
                                  node_color=[colors[i]] * len(component_nodes),
                                  node_size=600, alpha=0.8,
                                  label=f'Component {i+1} ({len(component)} nodes)')

        nx.draw_networkx_labels(subgraph, pos_full, font_size=9)
        nx.draw_networkx_edges(subgraph, pos_full, alpha=0.5, width=2)

        plt.title(f"Full Cluster {target_cluster_id} - All Components\n"
                  f"({subgraph.number_of_nodes()} traders total)",
                  fontsize=14, fontweight='bold')
        plt.legend()
        plt.axis('off')
        plt.tight_layout()
        plt.show()

else:
    print("No nodes to visualize in Cluster 3.")

# Additional analysis: Show all trading relationships in Cluster 3
if final_subgraph.number_of_edges() > 0:
    print(f"\nAll Trading Relationships in Cluster {target_cluster_id}:")
    edge_weights = [(u, v, d['weight']) for u, v, d in final_subgraph.edges(data=True)]
    edge_weights.sort(key=lambda x: x[2], reverse=True)

    for i, (u, v, weight) in enumerate(edge_weights):
        original_weight = weight * max_weight
        print(f"{i+1:2d}. {u} <-> {v}: {original_weight:,.0f} (normalized: {weight:.6f})")

    # Calculate total trading volume in this cluster
    total_volume = sum([weight * max_weight for _, _, weight in edge_weights])
    print(f"\nTotal trading volume in Cluster {target_cluster_id}: {total_volume:,.0f}")
    print(f"Average relationship strength: {total_volume/len(edge_weights):,.0f}")

else:
    print(f"No trading relationships found in Cluster {target_cluster_id}.")